In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import itertools
import copy
import random
import argparse
from IPython.display import Image
from database import create_database, save_database, load_database, update_database, print_database
from acquisition_functions import calc_entropy, calc_BALD, calc_var_rat, calc_Mean_STD, calc_uniform, get_TNC_preds, calc_var_rat_mod, mean_change
from dropout_CNN import CNN, train_w_acquisition, run_experiments
from MNISTdataset import get_balanced_set, get_dataset, get_indices, get_test_loader, set_seeds
from AI_layer import AILayer, run_AI_experiment
from MFVI_CNN import MFVI_CNN, MFVI_acquisition, run_MFVI_experiment
from feature_CNN import FeatureCNN, train_feature_CNN

In [5]:
# Loads relevant GPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Device: CUDA (NVIDIA GPU)")
else:
    device = torch.device("cpu")
    print("Device: CPU")

Device: MPS (Apple Silicon GPU)


In [ ]:
args = argparse.Namespace()
args.train_dataset = get_dataset()
args.train_indices, args.validation_indices, args.pool_indices = get_indices(args.train_dataset)
args.test_loader = get_test_loader()
args.device = torch.device("mps")
args.deterministic = False
args.T = 100
args.lr = 1e-3
args.wd = 1e-4
args.n_epochs = 50
args.sigma2 = 1.0
args.s2 = 1.0
args.batch_size = 128
args.num_classes = 10
args.retrain = True

args.n_acq = 100

In [ ]:
# Run acquisition experiments
acq_fns = {"entropy": calc_entropy, "uniform": calc_uniform, "BALD":calc_BALD, "var_rat": calc_var_rat,"Mean_STD": calc_Mean_STD, "var_rat_mod": calc_var_rat_mod,"mean_change":mean_change}

run_experiments(args=args, acq_fns = acq_fns, run_nums = [0,1,2], deterministic = False)

In [ ]:
# Initialise feature extractor for inference experiments
feature_CNN = FeatureCNN().to(args.device)
feature_CNN = train_feature_CNN(model = feature_CNN, 
                                train_dataset = args.train_dataset, 
                                train_indices = args.train_indices, 
                                device = args.device, n_epochs = 50)

In [ ]:
# Runs the analytic inference experiements
run_AI_experiment(args = args, run_nums = [0,1,2],feature_CNN = feature_CNN, retrain = True, train_reg = False)

In [ ]:

# Runs the MFVI experiments
run_MFVI_experiment(args = args, run_nums = [0,1,2], feature_CNN = feature_CNN, retrain = True)